**Workshop notebooks:** **01 — Matching & Loading (you are here)**&nbsp;·&nbsp;[02 — Overlay & Enrichment](https://colab.research.google.com/github/marlene-grabner/NetworkMedicine_Workshop/blob/main/02_overlay_enrichment.ipynb)&nbsp;·&nbsp;[03 — Bridging](https://colab.research.google.com/github/marlene-grabner/NetworkMedicine_Workshop/blob/main/03_bridging.ipynb)&nbsp;·&nbsp;[04 — Disease Modules (optional)](https://colab.research.google.com/github/marlene-grabner/NetworkMedicine_Workshop/blob/main/04_disease_modules_optional.ipynb)

# 01 — Matching & Loading

**Network Medicine Workshop · Kidney Disease · Part 1 of 3 (+ optional Part 4)**

This notebook handles the unglamorous-but-essential part of network medicine: getting a messy omics lists onto standardized IDs that match the node IDs used in the networks, and loading + sanity-checking those networks.

You're bringing **three independent datasets**, each from a different omics layer:
- **Transcripts** — DEGs from spatial transcriptomics
- **Proteins** — DE proteins from proteomics
- **Metabolites** — DE metabolites from metabolomics

Each gets matched to the ID system used by its corresponding network: transcripts and proteins both resolve to **NCBI Gene IDs** (transcripts → transcriptome network, proteins → PPI network), metabolites resolve to **KEGG Compound IDs** (→ metabolite network).

**What you'll do here (~15–20 min):**
1. Load all three DE lists
2. Batch-map gene/protein symbols → NCBI Gene IDs (with a fuzzy fallback for typos/old symbols)
3. Batch-map metabolite names → KEGG Compound IDs (local KEGG dictionary + vectorized fuzzy matching)
4. Load each network from your Drive `networks/` folder and print basic stats
5. Save everything to `processed/` so later notebooks can pick it up

**You will need, in your Drive workshop folder:**
- `data/DEGs.csv` — needs a `gene_symbol` column (transcripts)
- `data/DE_proteins.csv` — needs a `protein_symbol` column (the gene symbol encoding each protein; extra columns like fold-change kept)
- `data/DE_metabolites.csv` — needs a `metabolite_name` column
- `networks/ppi_network.csv` — edges `source,target[,weight]`, **NCBI Gene IDs**
- `networks/transcriptome_network.csv` — same format, **NCBI Gene IDs**
- `networks/metabolite_network.csv` — same format, **KEGG Compound IDs**

If your column names differ, just rename them below in the config cell — nothing else needs to change.


### How to use this notebook

This is a **Jupyter notebook**: a mix of text cells (like this one) and code cells that you run one at a time. You don't need Python installed on your own computer — everything runs in the cloud via **Google Colab**, for free, in your browser.

A few basics before you start:
- Click the ▶ play button on the left of a code cell (or select the cell and press **Shift+Enter**) to run it, then move to the next one.
- Run the cells **in order, top to bottom** — later cells depend on variables created by earlier ones.
- A cell is still running while you see a spinning circle next to it; wait for it to finish before moving on.
- If something goes wrong, you can always start fresh with **Runtime → Run all** in the Colab menu.
- Text cells like this one are just explanations — nothing to click, just read and move to the code cell below.

Don't worry about breaking anything: this notebook only reads your files and writes to your own Google Drive folder, so it's safe to experiment.

## Setup

Before we can do any actual network medicine, we need to get the environment ready: install a few Python packages that aren't part of Colab by default, connect to your Google Drive (so your data and results persist between sessions), and set the folder/file names we'll use throughout. Run the next few cells once, in order — you won't need to touch them again after that.

In [ ]:
# Install what we need. mygene = batch gene ID mapping, rapidfuzz = fast fuzzy string matching.
#!pip install -q mygene rapidfuzz networkx requests tqdm pandas


With the packages installed, the next step is connecting to your Google Drive. Colab's own storage is temporary — it gets wiped every time your session disconnects — so we use your Drive as a permanent place to keep the data files, the networks, and everything this notebook (and the ones after it) produces. Running the cell below pops up a Google sign-in / permission prompt; click through it and allow access so the rest of the notebook can read and write files there.

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')


Now that Drive is connected, this next cell defines all the file paths and settings the rest of the notebook uses — where your data lives, where the networks live, and where results get saved. It also creates those folders if they don't exist yet. You shouldn't need to change anything here unless your own CSV files use different column names than the defaults (`gene_symbol`, `protein_symbol`, `metabolite_name`) — if so, just edit the corresponding line below.

In [ ]:
# ---- CONFIG ----
import os, sys

GITHUB_REPO = "marlene-grabner/NetworkMedicine_Workshop"
GITHUB_BRANCH = "main"
GITHUB_RAW_BASE = f"https://raw.githubusercontent.com/{GITHUB_REPO}/{GITHUB_BRANCH}"

# Drive is just persistent scratch space between notebooks - nothing needs to be
# manually placed here, Step 0 below downloads everything into it automatically.
#BASE_DIR = "/content/drive/MyDrive/network_medicine_workshop"
BASE_DIR = "."

DATA_DIR = os.path.join(BASE_DIR, "data")
NET_DIR = os.path.join(BASE_DIR, "networks")
PROC_DIR = os.path.join(BASE_DIR, "processed")
HELPERS_DIR = os.path.join(BASE_DIR, "helpers")
for d in (DATA_DIR, NET_DIR, PROC_DIR, HELPERS_DIR):
    os.makedirs(d, exist_ok=True)
sys.path.insert(0, HELPERS_DIR)

DEG_FILE = os.path.join(DATA_DIR, "DEGs.csv")               # transcripts
PROTEIN_FILE = os.path.join(DATA_DIR, "DE_proteins.csv")    # proteins
METAB_FILE = os.path.join(DATA_DIR, "DE_metabolites.csv")   # metabolites

GENE_SYMBOL_COL = "gene_symbol"
PROTEIN_SYMBOL_COL = "protein_symbol"
METAB_NAME_COL = "metabolite_name"

# network layer -> (filename, which dataset maps onto it)
NETWORK_FILES = {
    "transcriptome": "transcriptomes_coexpression_kidney.tsv",   # matched against DEGs (transcripts), NCBI Gene IDs
    "ppi":           "ppi.tsv",             # matched against DE proteins, NCBI Gene IDs
    "metabolite":    "metabolite_network.tsv",      # matched against DE metabolites, KEGG Compound IDs
}
BRIDGE_FILE = "metabolite_gene_bridge.csv"           # used later, in Notebook 3

print("Base dir:", BASE_DIR)

## Step 0 — Fetch data & networks from GitHub

Everyone runs this. It downloads the workshop's data and network files — plus a few small helper
`.py` files the later notebooks import plotting code from — straight from the public GitHub
repository into your Drive. Nothing to upload by hand. Data and network files are skipped on
re-runs once downloaded (delete one if you want it re-fetched); the helper `.py` files are small
and still actively being tweaked, so those are always re-downloaded fresh, even on a re-run —
you'll never end up running stale plotting/analysis code without doing anything.

In [ ]:
import requests

HELPER_FILES = ["nb1_helpers.py", "nb2_helpers.py", "nb3_helpers.py", "nb4_helpers.py"]  # small
                                             # plotting/API-fetching modules the notebooks import from

FILES_TO_FETCH = {
    f"{GITHUB_RAW_BASE}/data/DEGs.csv": DEG_FILE,
    f"{GITHUB_RAW_BASE}/data/DE_proteins.csv": PROTEIN_FILE,
    f"{GITHUB_RAW_BASE}/data/DE_metabolites.csv": METAB_FILE,
    **{
        f"{GITHUB_RAW_BASE}/networks/{fname}": os.path.join(NET_DIR, fname)
        for fname in NETWORK_FILES.values()
    },
    f"{GITHUB_RAW_BASE}/networks/{BRIDGE_FILE}": os.path.join(NET_DIR, BRIDGE_FILE),
    **{
        f"{GITHUB_RAW_BASE}/helpers/{fname}": os.path.join(HELPERS_DIR, fname)
        for fname in HELPER_FILES
    },
}

for url, dest in FILES_TO_FETCH.items():
    # Helper .py files are small and still under active development, so always re-fetch them --
    # everything else (data/networks) is stable once downloaded, so those are cached as before.
    is_helper = os.path.basename(dest) in HELPER_FILES
    if os.path.exists(dest) and not is_helper:
        print(f"already have {os.path.basename(dest)} (delete it to re-download)")
        continue
    r = requests.get(url, timeout=60)
    if r.status_code == 200:
        with open(dest, "wb") as f:
            f.write(r.content)
        print(f"downloaded {os.path.basename(dest)}  ({len(r.content):,} bytes)")
    else:
        print(f"[warn] could not fetch {url} (status {r.status_code}) — "
              f"check GITHUB_REPO above and that the file exists in the repo yet")

## See what just landed on your Drive

Two ways to look at the downlaod we just did without leaving this tab:

- **Colab's file browser**: click the 📁 folder icon on the far left sidebar → `drive` → `MyDrive` → `network_medicine_workshop`. Double-click any CSV/TSV there and Colab opens a proper preview pane.
- Or just look at the printout below, which lists exactly what's there and how big each file is.

(You can also find the same folder by opening `drive.google.com` in another tab if you'd rather — it's a completely normal Drive folder, nothing Colab-specific.)


In [ ]:
print("Files now sitting in your Drive, under network_medicine_workshop/:\n")
for folder, label in [(DATA_DIR, "data/"), (NET_DIR, "networks/")]:
    print(label)
    for fname in sorted(os.listdir(folder)):
        size_kb = os.path.getsize(os.path.join(folder, fname)) / 1024
        print(f"  {fname}  ({size_kb:.1f} KB)")
    print()


Before letting pandas parse it, it's worth glancing at a file exactly as it downloaded — this is often the fastest way to spot a formatting surprise (wrong delimiter, an extra header row, stray quotes) before it causes a confusing error a few cells from now.

In [ ]:
# A raw peek at one of them, exactly as downloaded - before any Python touches it
print("First few lines of DEGs.csv, unprocessed:\n")
!head -n 5 {DEG_FILE}


## Step 1 — Load your three lists

Just load the data and take a quick look at each list. This is also your chance to catch obvious problems (wrong column names, encoding issues, duplicate entries) before they propagate downstream.


In [ ]:
import pandas as pd

degs = pd.read_csv(DEG_FILE)
proteins = pd.read_csv(PROTEIN_FILE)

# Metabolites are read as raw lines first, rather than straight into pandas, so we have full
# control over parsing before anything downstream depends on it.
with open(METAB_FILE, "r") as f:
    lines = [line.strip() for line in f if line.strip()]
header = lines[0].split(",")
metabolite_list = lines[1:]
metabs = pd.DataFrame(metabolite_list, columns=header).drop_duplicates()

print(f"Loaded {len(degs)} transcript rows, {len(proteins)} protein rows, {len(metabs)} metabolite rows (before cleanup)")


### A quick pass of data hygiene

Real-world spreadsheets almost always have small inconsistencies — a stray leading/trailing space, or the same entry listed twice. Left alone, these cause matching failures later that look like bugs but really are just `"ABCD1 "` (with a trailing space) not matching `"ABCD1"`. So before we do anything else, we strip whitespace from the identifier columns and drop exact duplicate rows.

In [ ]:
# Basic hygiene: strip whitespace, drop exact duplicates
degs[GENE_SYMBOL_COL] = degs[GENE_SYMBOL_COL].astype(str).str.strip()
proteins[PROTEIN_SYMBOL_COL] = proteins[PROTEIN_SYMBOL_COL].astype(str).str.strip()
metabs[METAB_NAME_COL] = metabs[METAB_NAME_COL].astype(str).str.strip()

degs = degs.drop_duplicates(subset=GENE_SYMBOL_COL).reset_index(drop=True)
proteins = proteins.drop_duplicates(subset=PROTEIN_SYMBOL_COL).reset_index(drop=True)
metabs = metabs.drop_duplicates(subset=METAB_NAME_COL).reset_index(drop=True)

print(f"Transcripts (DEGs):    {len(degs)} unique gene symbols")
print(f"Proteins (proteomics): {len(proteins)} unique protein/gene symbols")
print(f"Metabolites:           {len(metabs)} unique metabolite names")
degs.head()


Let's look at each table in turn to sanity-check it loaded correctly. First, transcripts (above) — now proteins:

In [ ]:
proteins.head()


And finally the metabolites list:

In [ ]:
metabs.head()


## Step 2 — Gene/protein symbol → NCBI Gene ID

Transcripts and proteins are both matched the same way, since both ultimately resolve to a gene identifier — we just do it as **one batch call per dataset** rather than looping symbol-by-symbol. `mygene.info`'s `querymany` accepts up to ~1000 identifiers per request and resolves current symbols *and* aliases/old names in a single pass. This is the single biggest speed difference vs. the naive approach.

Anything still unmatched after that gets a fuzzy-matching pass against the pool of symbols the batch call *did* resolve, so a typo like `"HAVCR-1"` still finds `HAVCR1`.

We wrap this in one reusable function and call it twice — once for transcripts, once for proteins — so both datasets get identical treatment.


In [ ]:
import mygene
from rapidfuzz import process, fuzz

mg = mygene.MyGeneInfo()

def match_symbols_to_ncbi(symbols, label):
    """Batch-map a list of gene/protein symbols to NCBI Gene IDs, with fuzzy fallback."""
    result = mg.querymany(
        symbols, scopes="symbol,alias", fields="entrezgene,symbol",
        species="human", returnall=True,
    )

    matched_rows = []
    seen = set()
    for hit in result["out"]:
        if "entrezgene" in hit and hit["query"] not in seen:
            matched_rows.append({
                "query": hit["query"],
                "ncbi_gene_id": str(hit["entrezgene"]),
                "matched_symbol": hit.get("symbol", ""),
                "match_type": "exact/alias",
                "match_score": 100,
            })
            seen.add(hit["query"])

    matches = pd.DataFrame(matched_rows).drop_duplicates(subset="query")
    missing = sorted(set(symbols) - set(matches["query"]))
    print(f"[{label}] matched directly: {len(matches)} / {len(symbols)}  |  unmatched: {len(missing)}")

    if missing:
        reference_symbols = matches["matched_symbol"].tolist()
        fuzzy_rows, still_missing = [], []
        for q in missing:
            best = process.extractOne(q, reference_symbols, scorer=fuzz.WRatio)
            if best and best[1] >= 85:
                row = matches.loc[matches["matched_symbol"] == best[0]].iloc[0]
                fuzzy_rows.append({
                    "query": q, "ncbi_gene_id": row["ncbi_gene_id"],
                    "matched_symbol": best[0], "match_type": "fuzzy", "match_score": best[1],
                })
            else:
                still_missing.append(q)
        matches = pd.concat([matches, pd.DataFrame(fuzzy_rows)], ignore_index=True)
        print(f"[{label}] recovered via fuzzy matching (score>=85): {len(fuzzy_rows)}  |  "
              f"genuinely unmatched: {len(still_missing)}")
        if still_missing:
            print(f"[{label}] unmatched examples:", still_missing[:15])

    return matches

gene_matches = match_symbols_to_ncbi(degs[GENE_SYMBOL_COL].tolist(), "transcripts")
protein_matches = match_symbols_to_ncbi(proteins[PROTEIN_SYMBOL_COL].tolist(), "proteins")


The batch call above already prints a running summary. It's still worth eyeballing the individual matches with the lowest confidence scores — a low fuzzy-match score can mean either a genuinely obscure alias that's actually correct, or a wrong match that's worth fixing by hand later. Sorting by score puts the ones most worth a second look at the top.

In [ ]:
gene_matches.sort_values("match_score").head(10)   # eyeball the shakiest transcript matches


Same check, now for the protein/gene symbols:

In [ ]:
protein_matches.sort_values("match_score").head(10)   # eyeball the shakiest protein matches


With both sets of matches in hand, we attach the NCBI Gene IDs (and match-quality info) back onto the original transcript and protein tables, so every downstream step has everything it needs in one place.

In [ ]:
degs_matched = degs.merge(
    gene_matches[["query", "ncbi_gene_id", "matched_symbol", "match_type", "match_score"]],
    left_on=GENE_SYMBOL_COL, right_on="query", how="left"
).drop(columns="query")

proteins_matched = proteins.merge(
    protein_matches[["query", "ncbi_gene_id", "matched_symbol", "match_type", "match_score"]],
    left_on=PROTEIN_SYMBOL_COL, right_on="query", how="left"
).drop(columns="query")

print(f"Transcripts: {degs_matched['ncbi_gene_id'].notna().sum()} / {len(degs_matched)} matched")
print(f"Proteins:    {proteins_matched['ncbi_gene_id'].notna().sum()} / {len(proteins_matched)} matched")


## Step 3 — Metabolite name → KEGG Compound ID

Same philosophy, different resource: don't ask an API for each metabolite. Instead we download the **full KEGG compound list once** (~19k entries, one request) and build a local name/synonym lookup. Exact matches resolve instantly; everything else goes through a fuzzy-matching pass (`rapidfuzz.process.cdist`), which compares all unmatched names against all KEGG names at once.

Matches are scored so you can see, at a glance, which ones need a human to double check. Standardizing data is often one of the most complicated parts of computational work as you need to handle the naming conventions of multiple people at once (brand names, plurals, stereochemistry prefixes, etc.).


In [ ]:
from nb1_helpers import fetch_kegg_compound_names   # the HTTP call + text parsing — see helpers/nb1_helpers.py

kegg_id_to_names, name_to_kegg = fetch_kegg_compound_names()
all_kegg_names = list(name_to_kegg.keys())
print(f"Loaded {len(kegg_id_to_names)} KEGG compounds")


First pass: exact (case-insensitive) matches against the KEGG name/synonym lookup we just built. This alone typically resolves the majority of common metabolite names.

In [ ]:
metab_names = metabs[METAB_NAME_COL].tolist()

exact_rows, unmatched_metabs = [], []
for name in metab_names:
    key = name.lower()
    if key in name_to_kegg:
        exact_rows.append({
            "query": name, "kegg_id": name_to_kegg[key],
            "matched_name": name, "match_type": "exact", "match_score": 100
        })
    else:
        unmatched_metabs.append(name)

print(f"Exact matches: {len(exact_rows)} / {len(metab_names)}")
print(f"Sent to fuzzy matching: {len(unmatched_metabs)}")


Second pass, for anything not resolved above: fuzzy string matching against the full KEGG name list, so a name like "D-Glucose" can still be linked to an entry recorded as "glucose" or with slightly different spelling, spacing, or capitalization.

In [ ]:
fuzzy_rows = []
if unmatched_metabs:
    import numpy as np
    # cdist = compute a full similarity matrix in one vectorized call (fast, C-backed)
    scores = process.cdist(unmatched_metabs, all_kegg_names, scorer=fuzz.WRatio, workers=-1)
    best_idx = scores.argmax(axis=1)
    best_scores = scores.max(axis=1)

    for name, idx, score in zip(unmatched_metabs, best_idx, best_scores):
        matched_name = all_kegg_names[idx]
        fuzzy_rows.append({
            "query": name, "kegg_id": name_to_kegg[matched_name],
            "matched_name": matched_name, "match_type": "fuzzy", "match_score": round(float(score), 1),
        })

metab_matches = pd.DataFrame(exact_rows + fuzzy_rows)

# Human-in-the-loop bands: >=90 auto-accept, 70-89 review, <70 likely wrong
metab_matches["review_flag"] = pd.cut(
    metab_matches["match_score"], bins=[0, 69.999, 89.999, 100],
    labels=["reject/manual", "review", "auto-accept"]
)
print(metab_matches["review_flag"].value_counts())
metab_matches.sort_values("match_score").head(15)


Fuzzy matches below a certain score are more likely to be wrong, so rather than accepting them automatically we flag them for a human to check. The cell below shows everything that isn't a confident auto-accept, and gives you a place to fix specific entries by hand if you spot an error.

In [ ]:
# --- Manual review step ---
# Anything flagged "review" or "reject/manual" below is worth a human glance.
# Edit the dict below to fix specific entries, then re-run this cell.
manual_overrides = {
    # e.g. "your messy name here": "C00031",
}

for name, cid in manual_overrides.items():
    mask = metab_matches["query"] == name
    metab_matches.loc[mask, ["kegg_id", "match_type", "match_score", "review_flag"]] = [cid, "manual", 100, "auto-accept"]

metab_matches[metab_matches["review_flag"] != "auto-accept"]


As with the genes and proteins earlier, we now attach the KEGG IDs (and match-quality info) back onto the metabolites table.

In [ ]:
metabs_matched = metabs.merge(
    metab_matches[["query", "kegg_id", "matched_name", "match_type", "match_score", "review_flag"]],
    left_on=METAB_NAME_COL, right_on="query", how="left"
).drop(columns="query")

print(f"{metabs_matched['kegg_id'].notna().sum()} / {len(metabs_matched)} metabolites matched to a KEGG ID")
metabs_matched.head()


## Step 4 — Load & screen the networks

Loading the edge lists of our networks and reporting the numbers: how many nodes, how many edges, how dense, how many connected components, and how big the largest one is. At up to ~20k nodes these are all quick operations in `networkx`.


In [ ]:
import networkx as nx
import time

graphs = {}
for layer, fname in NETWORK_FILES.items():
    path = os.path.join(NET_DIR, fname)
    G = nx.read_edgelist(path)
    
    t0 = time.time()
    graphs[layer] = G
    n_cc = nx.number_connected_components(G)
    largest_cc = len(max(nx.connected_components(G), key=len))
    print(f"[{layer}] {fname}")
    print(f"   nodes={G.number_of_nodes():,}  edges={G.number_of_edges():,}  "
          f"density={nx.density(G):.5f}  components={n_cc}  largest_cc={largest_cc:,} "
          f"({largest_cc/G.number_of_nodes():.1%} of nodes)")
    print(f"   loaded in {time.time()-t0:.1f}s\n")


**What to look for:** a healthy PPI/co-expression network usually has one dominant connected component covering the large majority of nodes. If your largest component is small relative to the total, either the network is unusually fragmented or something's off in the edge list (e.g. duplicate ID systems mixed together).


## Step 5 — Save everything for the next notebooks

Everything computed in this notebook — the three matched tables and the three loaded networks — gets written to your `processed/` folder on Drive. The next notebook picks these up directly, so you won't need to repeat any of this matching or loading. Networks are saved with Python's `pickle` (a format that preserves the exact graph object as-is), while the matched tables are saved as ordinary CSVs you can also open in Excel or Google Sheets if you want to inspect them by hand.

In [ ]:
import pickle

for layer, G in graphs.items():
    with open(os.path.join(PROC_DIR, f"graph_{layer}.pkl"), "wb") as f:
        pickle.dump(G, f)

degs_matched.to_csv(os.path.join(PROC_DIR, "degs_matched.csv"), index=False)
proteins_matched.to_csv(os.path.join(PROC_DIR, "proteins_matched.csv"), index=False)
metabs_matched.to_csv(os.path.join(PROC_DIR, "metabolites_matched.csv"), index=False)

print("Saved to", PROC_DIR)
print(os.listdir(PROC_DIR))


---
**Next:** open `02_overlay_enrichment.ipynb` — it picks up exactly where this notebook left off. There's also an **optional Notebook 4** (`04_disease_modules_optional.ipynb`) for a deeper dive into module significance and comparing your data against other diseases, if you have time for it.


**Workshop notebooks:** **01 — Matching & Loading (you are here)**&nbsp;·&nbsp;[02 — Overlay & Enrichment](https://colab.research.google.com/github/marlene-grabner/NetworkMedicine_Workshop/blob/main/02_overlay_enrichment.ipynb)&nbsp;·&nbsp;[03 — Bridging](https://colab.research.google.com/github/marlene-grabner/NetworkMedicine_Workshop/blob/main/03_bridging.ipynb)&nbsp;·&nbsp;[04 — Disease Modules (optional)](https://colab.research.google.com/github/marlene-grabner/NetworkMedicine_Workshop/blob/main/04_disease_modules_optional.ipynb)